# FraudShield AI — Exploratory Data Analysis

This notebook explores the synthetic transaction dataset used by FraudShield AI:
class balance, feature distributions, and how fraud differs from legitimate
activity. Run it top-to-bottom (`Kernel > Restart & Run All`).

> Uses the project's own `src.data_generator`, so no external data is needed.
> Run from the repository root so the `src` package is importable.

In [ ]:
import sys, os
# Make the project root importable when the notebook lives in notebooks/.
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_generator import generate_transactions

df = generate_transactions(n_samples=20_000, fraud_ratio=0.04, random_state=42)
df.head()

## 1. Class balance

Fraud is rare — this imbalance is why we use `class_weight="balanced"`.

In [ ]:
counts = df['is_fraud'].value_counts().sort_index()
print(counts)
print(f"Fraud rate: {df['is_fraud'].mean():.2%}")

counts.plot(kind='bar', color=['#2ca02c', '#d62728'])
plt.xticks([0, 1], ['legit', 'fraud'], rotation=0)
plt.title('Class balance')
plt.ylabel('count')
plt.show()

## 2. Amount distribution by class

Fraudulent transactions skew toward larger amounts, but the low end overlaps
with legitimate spending (card-testing fraud is small).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bins = np.linspace(0, df['amount'].quantile(0.99), 50)
ax.hist(df[df.is_fraud == 0]['amount'], bins=bins, alpha=0.6, density=True, label='legit', color='#1f77b4')
ax.hist(df[df.is_fraud == 1]['amount'], bins=bins, alpha=0.6, density=True, label='fraud', color='#d62728')
ax.set_xlabel('amount'); ax.set_ylabel('density'); ax.legend(); ax.set_title('Amount by class')
plt.show()

## 3. Fraud rate by hour of day

Fraud concentrates in the small hours of the night.

In [ ]:
by_hour = df.groupby('hour')['is_fraud'].mean()
by_hour.plot(kind='bar', color='#ff7f0e', figsize=(9, 3))
plt.ylabel('fraud rate'); plt.title('Fraud rate by hour of day')
plt.show()

## 4. Fraud rate across categorical features

In [ ]:
for col in ['merchant_category', 'device_type', 'foreign_transaction', 'is_new_device']:
    print(f"\n=== fraud rate by {col} ===")
    print(df.groupby(col)['is_fraud'].mean().sort_values(ascending=False))

## 5. Numeric feature correlation with the label

In [ ]:
numeric = df.select_dtypes(include='number').drop(columns=['transaction_id'])
corr = numeric.corr()['is_fraud'].drop('is_fraud').sort_values()
corr.plot(kind='barh', figsize=(6, 4), color='#2ca02c')
plt.title('Correlation with is_fraud'); plt.xlabel('Pearson r')
plt.show()
corr

## Takeaways

- The dataset is **imbalanced** (~4% fraud).
- **Account age**, **transaction velocity**, **amount** and **hour** carry the
  strongest fraud signal — matching the model's feature importances.
- Classes **overlap** (by design), so the task is non-trivial, just like real life.

Next: train the model with `python -m src.train` and view the charts with
`python -m src.visualize`.